<a href="https://colab.research.google.com/github/AzaharaAED/Proyecto_ecosistema/blob/main/nevhor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>












## MODELO DETECCIÓN ALIMENTOS NEVERA

1. Qwen simple
2. Qwen con SAM para detectar la cantidad de cada alimentos pero ñeh
3. Qwen por cuadrículas

Para la estimación de la cantidad de alimentos se optó por un enfoque basado en división de la imagen en cuadrículas, en lugar de utilizar un modelo de segmentación. En primer lugar, el modelo multimodal Qwen2-VL se aplicó sobre la imagen completa para obtener una lista global de alimentos visibles. En este caso, el modelo identificó los siguientes elementos: tomato, pepper, lettuce, egg, cheese y milk.

Posteriormente, la imagen se dividió en una rejilla de 6×6 cuadrículas (36 regiones). Cada una de estas regiones se analizó de forma independiente con el modelo Qwen para identificar si contenía algún alimento. A partir de estos resultados se contabilizó el número de cuadrículas en las que aparece cada alimento, lo que permite estimar de forma aproximada el espacio visual que ocupa dentro de la imagen.

Los resultados muestran que lettuce aparece en 9 cuadrículas, lo que indica una presencia elevada en la imagen y se traduce en una cantidad high. Por su parte, tomato aparece en 3 cuadrículas, clasificándose como medium, mientras que pepper y milk aparecen en una sola cuadrícula, por lo que se clasifican como low. Finalmente, aunque egg y cheese fueron detectados en la fase global, no aparecieron en ninguna cuadrícula analizada, por lo que su cantidad se mantiene como unknown.

En conjunto, este enfoque basado en cuadrículas permite estimar cantidades de forma más estable e interpretable, ya que la cantidad se calcula directamente a partir del número de regiones de la imagen donde aparece cada alimento. Esto facilita la interpretación de los resultados y evita algunos problemas asociados a los modelos de segmentación automática, como la fragmentación irregular de los objetos.

In [ ]:
import re
from pathlib import Path

import torch
from PIL import Image, UnidentifiedImageError
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

# CONFIGURACIÓN GLOBAL

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QWEN_MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

_qwen_processor = None
_qwen_model = None

# CARGA DIFERIDA DEL MODELO

def get_qwen_model_and_processor():
    """
    Carga el modelo solo la primera vez.
    Así al hacer import nevera NO se descarga ni se inicializa nada.
    """
    global _qwen_processor, _qwen_model

    if _qwen_processor is None:
        _qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_NAME)

    if _qwen_model is None:
        _qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )

    return _qwen_processor, _qwen_model

# 1. CARGA DE IMAGEN DESDE RUTA LOCAL

def load_image_from_path(image_path, resize_half=True):
    """
    Carga una imagen desde ruta local.
    En Colab, si el archivo está en el panel de la izquierda,
    normalmente basta con usar: 'nevera.jpeg'
    """
    if not isinstance(image_path, str):
        raise TypeError("image_path debe ser un string")

    path = Path(image_path).expanduser()

    if not path.is_absolute():
        path = Path.cwd() / path

    if not path.exists():
        archivos_disponibles = [p.name for p in Path.cwd().iterdir() if p.is_file()]
        raise ValueError(
            f"No se encontró la imagen en la ruta: {path}\n"
            f"Directorio actual: {Path.cwd()}\n"
            f"Archivos disponibles aquí: {archivos_disponibles}"
        )

    if not path.is_file():
        raise ValueError(f"La ruta no apunta a un archivo: {path}")

    try:
        with Image.open(path) as img:
            img.verify()

        image = Image.open(path).convert("RGB")
    except UnidentifiedImageError:
        raise ValueError(f"El archivo no es una imagen válida: {path}")
    except OSError as e:
        raise ValueError(f"No se pudo abrir la imagen '{path}': {e}")

    if resize_half:
        new_w = max(1, image.width // 2)
        new_h = max(1, image.height // 2)
        image = image.resize((new_w, new_h))

    return image

# 2. CONSULTA A QWEN

def ask_qwen_single_image(img, prompt, max_new_tokens=80):
    processor, model = get_qwen_model_and_processor()

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text_prompt],
        images=[img],
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, output_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )[0]

    return output_text.strip()

# 3. LIMPIEZA DE TEXTO

INVALID_FOOD_NAMES = {
    "", "food", "food_name", "name_of_food", "visible food", "visible foods",
    "item", "items", "object", "objects", "ingredient", "ingredients",
    "unknown", "not_food", "none", "n/a"
}


def normalize_food_name(name):
    if name is None:
        return None

    name = name.strip().lower()
    name = re.sub(r"[^a-zA-Z\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()

    if name in INVALID_FOOD_NAMES:
        return None

    replacements = {
        "tomatoes": "tomato",
        "cherry tomatoes": "tomato",
        "bell peppers": "pepper",
        "bell pepper": "pepper",
        "peppers": "pepper",
        "eggs": "egg",
        "cucumbers": "cucumber",
        "carrots": "carrot",
        "yogurts": "yogurt",
        "limes": "lime",
        "lettuces": "lettuce",
        "broccolis": "broccoli",
        "cheeses": "cheese",
    }

    if name in replacements:
        return replacements[name]

    if name.endswith("ies") and len(name) > 4:
        name = name[:-3] + "y"
    elif name.endswith("oes") and len(name) > 4:
        name = name[:-2]
    elif name.endswith("s") and len(name) > 3 and not name.endswith("ss"):
        name = name[:-1]

    if name in replacements:
        name = replacements[name]

    if name in INVALID_FOOD_NAMES:
        return None

    return name


def clean_global_foods(raw_text):
    parts = [x.strip() for x in raw_text.split(",") if x.strip()]
    cleaned = []
    seen = set()

    for p in parts:
        norm = normalize_food_name(p)
        if norm is None:
            continue
        if norm not in seen:
            seen.add(norm)
            cleaned.append(norm)

    return cleaned


def parse_grid_food(raw_text):
    txt = raw_text.strip().lower()

    if txt == "not_food":
        return None

    txt = txt.replace("\n", " ").strip()
    txt = txt.split(",")[0].strip()
    txt = txt.split("-")[0].strip()

    return normalize_food_name(txt)

# 4. DETECCIÓN GLOBAL

def detectar_global_foods(image):
    global_prompt = """
Look at the full image and identify the visible foods.

Return only a comma-separated list of UNIQUE food names.
Do not explain anything.
Do not repeat any food.
Do not include placeholders.
Do not hallucinate foods that are not clearly visible.
Maximum 12 unique foods.

Examples of valid output:
tomato, pepper, lettuce, egg, cheese, milk
"""

    global_result = ask_qwen_single_image(image, global_prompt, max_new_tokens=60)
    global_foods = clean_global_foods(global_result)

    return {
        "raw_result": global_result,
        "global_foods": global_foods
    }

# 5. CREAR GRID

def crear_grid(image, grid_rows=6, grid_cols=6):
    img_w, img_h = image.size
    cell_w = img_w // grid_cols
    cell_h = img_h // grid_rows

    grid_cells = []

    for r in range(grid_rows):
        for c in range(grid_cols):
            x1 = c * cell_w
            y1 = r * cell_h
            x2 = (c + 1) * cell_w if c < grid_cols - 1 else img_w
            y2 = (r + 1) * cell_h if r < grid_rows - 1 else img_h

            crop = image.crop((x1, y1, x2, y2))

            grid_cells.append({
                "row": r,
                "col": c,
                "box": [x1, y1, x2, y2],
                "crop": crop
            })

    return grid_cells, cell_w, cell_h

# 6. ANÁLISIS DE CELDAS

def analizar_grid_cells(grid_cells):
    grid_prompt = """
Identify the main visible food in this image region.

Return ONLY one food name.
Examples: tomato, pepper, lettuce, egg, cheese, milk
If there is no clear food visible, return exactly: not_food

Rules:
- Return only one food name.
- Do not explain anything.
- Do not use placeholders.
- Do not mention packaging or container type.
"""

    grid_results = []

    for cell in grid_cells:
        result = ask_qwen_single_image(cell["crop"], grid_prompt, max_new_tokens=8)
        food_name = parse_grid_food(result)

        grid_results.append({
            "row": cell["row"],
            "col": cell["col"],
            "box": cell["box"],
            "raw_result": result,
            "food": food_name
        })

    return grid_results

# 7. CONTAR ALIMENTOS EN GRID

def contar_foods_en_grid(grid_results, global_foods=None):
    if global_foods is None:
        global_foods = []

    food_grid_counts = {}

    for item in grid_results:
        food = item["food"]
        if food is None:
            continue

        if len(global_foods) > 0 and food not in global_foods:
            continue

        if food not in food_grid_counts:
            food_grid_counts[food] = {
                "count_cells": 1,
                "boxes": [item["box"]]
            }
        else:
            food_grid_counts[food]["count_cells"] += 1
            food_grid_counts[food]["boxes"].append(item["box"])

    return food_grid_counts

# 8. CONVERTIR CONTEO A NIVEL

def grid_count_to_level(count_cells, total_cells):
    ratio = count_cells / total_cells

    if ratio <= 0.05:
        return "low"
    elif ratio <= 0.20:
        return "medium"
    else:
        return "high"


def fusionar_resultados(global_foods, food_grid_counts, grid_rows=6, grid_cols=6):
    total_cells = grid_rows * grid_cols
    final_food_dict = {}

    for food in global_foods:
        final_food_dict[food] = {
            "food": food,
            "amount_level": "unknown",
            "count_cells": 0,
            "boxes": []
        }

    for food, stats in food_grid_counts.items():
        level = grid_count_to_level(stats["count_cells"], total_cells)

        if food not in final_food_dict:
            final_food_dict[food] = {
                "food": food,
                "amount_level": level,
                "count_cells": stats["count_cells"],
                "boxes": stats["boxes"]
            }
        else:
            final_food_dict[food]["amount_level"] = level
            final_food_dict[food]["count_cells"] = stats["count_cells"]
            final_food_dict[food]["boxes"] = stats["boxes"]

    return list(final_food_dict.values())

# 9. FUNCIÓN PRINCIPAL DEL MÓDULO

def detectar_alimentos_nevera(image_path="nevera.jpeg", grid_rows=6, grid_cols=6, resize_half=True):
    image = load_image_from_path(image_path, resize_half=resize_half)

    global_data = detectar_global_foods(image)
    global_foods = global_data["global_foods"]

    grid_cells, _, _ = crear_grid(image, grid_rows=grid_rows, grid_cols=grid_cols)
    grid_results = analizar_grid_cells(grid_cells)
    food_grid_counts = contar_foods_en_grid(grid_results, global_foods=global_foods)

    final_foods = fusionar_resultados(
        global_foods,
        food_grid_counts,
        grid_rows=grid_rows,
        grid_cols=grid_cols
    )

    plain_text = [f"{item['food']} - {item['amount_level']}" for item in final_foods]

    return {
        "image": image,
        "global_raw_result": global_data["raw_result"],
        "global_foods": global_foods,
        "grid_results": grid_results,
        "food_grid_counts": food_grid_counts,
        "final_foods": final_foods,
        "plain_text": plain_text
    }


def obtener_ingredientes_nevera_para_modelo(image_path="nevera.jpeg", grid_rows=6, grid_cols=6, resize_half=True):
    resultado = detectar_alimentos_nevera(
        image_path=image_path,
        grid_rows=grid_rows,
        grid_cols=grid_cols,
        resize_half=resize_half
    )

    ingredientes = [item["food"] for item in resultado["final_foods"]]
    cantidades = {
        item["food"]: item["amount_level"]
        for item in resultado["final_foods"]
    }

    return {
        "ingredientes": ingredientes,
        "cantidades": cantidades
    }

## MODELO DETECCIÓN PLATOS HORNO Y AF

In [ ]:
# 1. CARGAR IMAGEN DESDE RUTA LOCAL

def cargar_imagen_desde_ruta(image_path, resize_half=True):
    try:
        image = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        raise ValueError(f"No se encontró la imagen en la ruta: {image_path}")
    except UnidentifiedImageError:
        raise ValueError(f"El archivo no es una imagen válida: {image_path}")

    if resize_half:
        new_w = max(1, image.width // 2)
        new_h = max(1, image.height // 2)
        image = image.resize((new_w, new_h))

    return image

# 2. LIMPIAR SALIDA

def limpiar_salida_plato(texto):
    if texto is None:
        return None

    texto = texto.strip().lower()
    texto = texto.replace("\n", " ")
    texto = " ".join(texto.split())

    if texto in ["", "not food", "not_food", "no food"]:
        return "not food"

    return texto

# 3. INFERENCIA SOBRE UNA IMAGEN

def detectar_plato_imagen(image, max_new_tokens=20):
    processor, model = get_qwen_model_and_processor()

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": """Identify the prepared dish shown in the image.

Rules:
- If you can clearly identify the dish, answer with only the dish name.
- If you cannot identify the exact dish, answer with only the main visible ingredients.
- Do not explain.
- Do not use phrases like "it looks like", "maybe", or "I think".
- Do not write full sentences.
- Output only one short line.
- If there are several foods, mention only the main dish.
- If this is not food, answer only: not food.
"""
                },
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text_prompt],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, output_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )[0]

    return limpiar_salida_plato(output_text)

# 4. FUNCIÓN PRINCIPAL

def detectar_plato_desde_ruta(image_path, resize_half=True, max_new_tokens=20):
    image = cargar_imagen_desde_ruta(image_path, resize_half=resize_half)
    plato = detectar_plato_imagen(image, max_new_tokens=max_new_tokens)

    return {
        "image": image,
        "plato": plato,
        "raw_output": plato
    }

# 5. FUNCIÓN FINAL PARA EL MODELO GENERAL

def obtener_plato_para_modelo(image_path, resize_half=True, max_new_tokens=20):
    resultado = detectar_plato_desde_ruta(
        image_path=image_path,
        resize_half=resize_half,
        max_new_tokens=max_new_tokens
    )

    plato = resultado["plato"]

    if plato == "not food" or plato is None:
        return {
            "plato": None,
            "es_comida": False
        }

    return {
        "plato": plato,
        "es_comida": True
    }